# 07 - Native HDF5 Data Management

This notebook is hardware-independent. It shows the current storage module used by `ExperimentData.save()`:

- one experiment per HDF5 file
- raw complex IQ stored natively
- axes and dataset dimensions stored with metadata
- a rebuildable `catalog.sqlite` search index
- lightweight inspection and validation utilities

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import numpy as np

from QickworkspaceV2 import ExperimentData, QualityFlag
from QickworkspaceV2.tools.hdf5_store import (
    find_experiments,
    inspect_file,
    load_result,
    rebuild_catalog,
    validate_file,
)

DATA_ROOT = Path('data_native').resolve()
DATA_ROOT.mkdir(exist_ok=True)
DATA_ROOT

## 1. Create a few offline experiment files

Real experiment runs already return `ExperimentData`. For a portable tutorial we create small synthetic results with the same public fields and save them through the same API.

In [ ]:
def make_result(experiment_type, qubit, x, y, *, session_id, tags):
    raw_iq = y + 0.03j * np.sin(np.linspace(0, np.pi, len(x)))
    return ExperimentData(
        experiment_type=experiment_type,
        raw_iq=raw_iq,
        x_axis=x,
        axes={
            'x': {
                'values': x,
                'label': 'Delay' if experiment_type == 't1' else 'Drive frequency',
                'unit': 'us' if experiment_type == 't1' else 'MHz',
            },
        },
        dataset_dims={'iq': ['x']},
        fit_params=np.array([float(np.max(y)), float(np.min(y))]),
        fit_errors=np.array([0.01, 0.01]),
        fit_result={'amplitude': (float(np.ptp(y)), 0.01)},
        scalar_result=float(x[np.argmin(y)]),
        quality=QualityFlag.GOOD,
        quality_message='synthetic tutorial data',
        metadata={'qubit': qubit, 'operator': 'tutorial'},
        session_id=session_id,
        tags=list(tags),
    )

# Use unique filenames so ExperimentData.save() never overwrites old runs.
session_id = 'tutorial-session'
x_spec = np.linspace(6500, 6600, 101)
x_t1 = np.linspace(0, 80, 81)
results = [
    make_result('resonator_spec', 'Q1', x_spec, 1.0 - 0.35*np.exp(-((x_spec-6551)/9)**2), session_id=session_id, tags=['tutorial', 'spec']),
    make_result('t1', 'Q1', x_t1, 0.9*np.exp(-x_t1/28) + 0.08, session_id=session_id, tags=['tutorial', 'coherence']),
    make_result('resonator_spec', 'Q2', x_spec, 1.0 - 0.22*np.exp(-((x_spec-6534)/12)**2), session_id=session_id, tags=['tutorial', 'spec']),
]

paths = [result.save(data_root=DATA_ROOT, comment='generated by tutorial 07') for result in results]
paths

## 2. Search the rebuildable catalog

Saving with `catalog=True` updates `catalog.sqlite`. Search it directly during normal work. If the catalog is deleted or files are moved into the folder later, rebuild it from the HDF5 file metadata.

In [ ]:
# ExperimentData.save() registers every file in catalog.sqlite automatically.
refs = find_experiments(qubit='Q1', tags=['tutorial'], limit=10, data_root=DATA_ROOT)
print(f'Catalog matches for Q1: {len(refs)}')
for ref in refs:
    print(ref.timestamp_local, ref.experiment_type, ref.qubits, ref.quality, ref.path.name)

# Use rebuild_catalog(DATA_ROOT) only after moving files or deleting catalog.sqlite.


## 3. Inspect and validate one file

`inspect_file()` reads lightweight metadata. `validate_file()` checks schema completeness and verifies that datasets with `dims` attributes match the named axes.

In [ ]:
ref = refs[0]
info = inspect_file(ref.path)
report = validate_file(ref.path)

print('experiment_id:', info['experiment_id'])
print('type:', info['experiment_type'])
print('qubits:', info['qubits'])
print('tags:', info['tags'])
print('comment:', info['comment'])
print('valid:', report.valid)
print('warnings:', report.warnings)
print('errors:', report.errors)

## 4. Load the full result

Use either `ExperimentData.load(path)` or `load_result(path)`. `ExperimentReference.load()` is a convenient lazy loader for entries returned by `find_experiments()`.

In [ ]:
loaded = ref.load()
print(loaded)
print('scalar_result:', loaded.scalar_result)
print('x axis points:', len(loaded.x_axis) if loaded.x_axis is not None else None)
print('raw_iq dtype:', np.asarray(loaded.raw_iq).dtype)
print('dataset dims:', loaded.dataset_dims)

In [ ]:
import matplotlib.pyplot as plt

x = loaded.x_axis
raw = np.asarray(loaded.raw_iq)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(x, np.abs(raw), marker='.', ms=3)
ax.set_xlabel(f"{loaded.x_name or 'x'} ({loaded.x_unit or 'a.u.'})")
ax.set_ylabel('|IQ|')
ax.set_title(f'{loaded.experiment_type} - {loaded.metadata.get("qubit", "unknown qubit")}')
ax.grid(alpha=0.25)
fig.tight_layout()

## 5. Use the local HDF5 viewer

The repository also includes a local-only FastAPI viewer for browsing catalog rows, inspecting metadata, validating files, and plotting selected raw/analysis datasets.

From the repository root:

```bash
python hdf5_viewer/hdf5_viewer_server.py
```

Then open `http://127.0.0.1:8000` and choose the data folder. On Windows you can also run `hdf5_viewer/start_hdf5_viewer.bat`.

## 6. Data folder layout

Native saves create date-organized files when no explicit path is supplied:

```text
data_root/
  catalog.sqlite
  2026/
    08/
      Data_0803/
        s002_res_ge_Q1_<experiment_id>.hdf5
```

The SQLite file is only a search index. The HDF5 files are the source of truth and can rebuild the catalog at any time with `rebuild_catalog(data_root)`.